# Module 1 Homework: Yellow Taxi Trip Records

This notebook solves the Module 1 homework assignment by analyzing Yellow Taxi Trip Records for January and February 2023.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
import os
import urllib.request

# Download January 2023 data
url_jan = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
file_jan = "yellow_tripdata_2023-01.parquet"

if os.path.exists(file_jan):
    print(f"January data already exists: {file_jan}")
else:
    print(f"Downloading January data...")
    urllib.request.urlretrieve(url_jan, file_jan)

df_jan = pd.read_parquet(file_jan)

# Download February 2023 data
url_feb = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"
file_feb = "yellow_tripdata_2023-02.parquet"

if os.path.exists(file_feb):
    print(f"February data already exists: {file_feb}")
else:
    print(f"Downloading February data...")
    urllib.request.urlretrieve(url_feb, file_feb)

df_feb = pd.read_parquet(file_feb)

## Q1: Count number of columns in January data

In [3]:
# Q1
n_columns = df_jan.shape[1]
print(f"Q1: Number of columns in January data: {n_columns}")

Q1: Number of columns in January data: 19


## Q2: Compute standard deviation of duration (in minutes)

In [4]:
# Q2
# Calculate trip duration in minutes
df_jan['duration'] = (df_jan['tpep_dropoff_datetime'] - df_jan['tpep_pickup_datetime']).dt.total_seconds() / 60

duration_std = df_jan['duration'].std()
print(f"Q2: Standard deviation of duration (minutes): {duration_std:.4f}")

Q2: Standard deviation of duration (minutes): 42.5944


## Q3: Filter outliers (duration 1-60 min) and compute fraction remaining

In [5]:
# Q3
n_original = len(df_jan)

# Filter outliers: keep only trips with duration between 1 and 60 minutes
df_jan_filtered = df_jan[(df_jan['duration'] >= 1) & (df_jan['duration'] <= 60)]

n_filtered = len(df_jan_filtered)
fraction_remaining = n_filtered / n_original

print(f"Q3: Fraction of trips remaining after filtering outliers: {fraction_remaining:.4f}")

Q3: Fraction of trips remaining after filtering outliers: 0.9812


## Q4: One-hot encode PULocationID and DOLocationID, report feature matrix dimensionality

In [7]:
# Q4
# Work with the filtered dataframe
df = df_jan_filtered.copy()

# Convert location IDs to strings for DictVectorizer
df['PULocationID'] = df['PULocationID'].astype(str)
df['DOLocationID'] = df['DOLocationID'].astype(str)

# Create list of dictionaries for DictVectorizer
train_dicts = df[['PULocationID', 'DOLocationID']].to_dict(orient='records')

# Apply DictVectorizer for one-hot encoding
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

dimensionality = X_train.shape[1]
print(f"Q4: Feature matrix dimensionality after one-hot encoding: {dimensionality}")

Q4: Feature matrix dimensionality after one-hot encoding: 515


## Q5: Train LinearRegression, report RMSE on training data

In [8]:
# Q5
# Target variable
y_train = df['duration'].values

# Train Linear Regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict on training data
y_train_pred = lr.predict(X_train)

# Calculate RMSE on training data
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
print(f"Q5: RMSE on training data: {rmse_train:.4f}")

Q5: RMSE on training data: 7.6493


## Q6: Apply model to February data, report RMSE on validation

In [9]:
# Q6
# Prepare February data
df_feb['duration'] = (df_feb['tpep_dropoff_datetime'] - df_feb['tpep_pickup_datetime']).dt.total_seconds() / 60

# Filter outliers in February data
df_feb_filtered = df_feb[(df_feb['duration'] >= 1) & (df_feb['duration'] <= 60)]

# Convert location IDs to strings
df_feb_filtered['PULocationID'] = df_feb_filtered['PULocationID'].astype(str)
df_feb_filtered['DOLocationID'] = df_feb_filtered['DOLocationID'].astype(str)

# Create validation dictionaries and transform using fitted DictVectorizer
val_dicts = df_feb_filtered[['PULocationID', 'DOLocationID']].to_dict(orient='records')
X_val = dv.transform(val_dicts)

# Target variable for validation
y_val = df_feb_filtered['duration'].values

# Predict on validation data
y_val_pred = lr.predict(X_val)

# Calculate RMSE on validation data
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
print(f"Q6: RMSE on validation (February) data: {rmse_val:.4f}")

C:\Users\Alfa\AppData\Local\Temp\ipykernel_6340\2460938326.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_feb_filtered['PULocationID'] = df_feb_filtered['PULocationID'].astype(str)
C:\Users\Alfa\AppData\Local\Temp\ipykernel_6340\2460938326.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_feb_filtered['DOLocationID'] = df_feb_filtered['DOLocationID'].astype(str)


Q6: RMSE on validation (February) data: 7.8118
